# 08 - Heterogeneous Scholarly Evidence Graph (item 5)

Builds a {paper, method, dataset} heterogeneous graph over the benchmark's abstracts (heuristic lexicon + acronym-pattern entity extraction -- see `entities.py`; NOT LLM-based, deliberately, for zero-cost determinism), trains a small hand-rolled 2-layer relational GNN (`hetero_graph.py`'s `RGCNLite` -- no torch_geometric/DGL dependency, for Kaggle-image portability) via link prediction on silver-supported paper pairs, then adds the resulting embedding-cosine-similarity as one extra feature on top of item 1's 16-feature verifier and checks whether it improves gold F1.

**Read `hetero_graph.py`'s module docstring before citing these numbers** -- three explicit scoping cuts (heuristic extraction, no task nodes, hand-rolled GNN) are documented there, each with its cost/benefit reasoning.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
import json
import numpy as np
import torch
from graph_utils import CitationGraph, load_relational_claims, verify
from hard_negatives import build_tfidf_index
from verifier_features import build_dataset
import verifier_model as vm
from hetero_graph import (build_hetero_graph, positive_pairs_from_claims, _to_node_id_pairs,
    train_rgcn, hetero_pair_similarity)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

graph = CitationGraph.load(REFS_CACHE)
all_claims = load_relational_claims(BENCH, CITE_RESULTS)

paper_abstracts = {}
with open(BENCH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for gt in rec.get("ground_truth_papers") or []:
            pid, ab = gt.get("paper_id"), gt.get("abstract")
            if pid and ab:
                paper_abstracts[pid] = ab
print(f"{len(paper_abstracts)} papers with abstracts")
tfidf_vectors = build_tfidf_index(paper_abstracts)

print("Building heterogeneous graph...")
hg, entity_cache = build_hetero_graph(paper_abstracts, graph)
n_methods = sum(1 for t in hg.node_type if t == "method")
n_datasets = sum(1 for t in hg.node_type if t == "dataset")
print(f"nodes: {hg.n_nodes} (papers={len(paper_abstracts)}, methods={n_methods}, datasets={n_datasets})")
print(f"edges: {len(hg.edges)}")

silver = vm.load_silver_labels(CITE_RESULTS)
gold = vm.load_gold_labels(BENCH_VERIFIED) if BENCH_VERIFIED else {}
silver_clean = {k: v for k, v in silver.items() if k not in gold}

pos_pairs = positive_pairs_from_claims(all_claims, silver_clean, paper_abstracts)
pos_node_pairs = _to_node_id_pairs(hg, pos_pairs)
print(f"positive (silver-supported) paper-pair edges for link prediction: {len(pos_node_pairs)}")

print("\nTraining R-GCN-lite (link prediction)...")
model, embeddings = train_rgcn(hg, pos_node_pairs, n_epochs=30, seed=0)

X_train, y_train, meta_train = build_dataset(all_claims, graph, tfidf_vectors, silver_clean)
X_gold, y_gold, meta_gold = build_dataset(all_claims, graph, tfidf_vectors, gold)

def add_hetero_col(X, meta):
    extra = np.array([[hetero_pair_similarity(m["paper_a"], m["paper_b"], hg, embeddings)] for m in meta],
                      dtype=np.float32) if meta else np.zeros((0, 1), dtype=np.float32)
    return np.concatenate([X, extra], axis=1)

X_train_h = add_hetero_col(X_train, meta_train)
X_gold_h = add_hetero_col(X_gold, meta_gold)

results = {}
rule_preds = np.array([1 if verify(m["paper_a"], m["relation"], m["paper_b"], graph) == "supported" else 0
                       for m in meta_gold])
results["rule_based"] = vm.bootstrap_prf1(y_gold, rule_preds)

scaler = StandardScaler().fit(X_train)
lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(scaler.transform(X_train), y_train)
lr_preds = lr.predict(scaler.transform(X_gold))
results["logreg_baseline_16feat"] = vm.bootstrap_prf1(y_gold, lr_preds)

scaler_h = StandardScaler().fit(X_train_h)
lr_h = LogisticRegression(max_iter=2000, class_weight="balanced").fit(scaler_h.transform(X_train_h), y_train)
lr_h_preds = lr_h.predict(scaler_h.transform(X_gold_h))
results["logreg_plus_hetero_17feat"] = vm.bootstrap_prf1(y_gold, lr_h_preds)

for name, r in results.items():
    print(f"{name}: precision={r['precision']:.3f} recall={r['recall']:.3f} f1={r['f1']:.3f} "
          f"(F1 CI [{r['f1_ci'][0]:.3f}, {r['f1_ci'][1]:.3f}])")

# Paired comparison: same gold claims scored under both, more sensitive than
# eyeballing the two CIs above -- the direct test of whether the hetero-graph
# feature actually helps (see verifier_model.paired_bootstrap_diff's docstring).
hetero_cmp = vm.paired_bootstrap_diff(y_gold, lr_h_preds, lr_preds)
results["paired_plus_hetero_vs_baseline"] = hetero_cmp
sig = " *SIGNIFICANT*" if hetero_cmp["f1_significant"] else ""
print(f"\nPaired +hetero-vs-baseline: F1 diff={hetero_cmp['f1_diff']:+.3f} "
      f"CI=[{hetero_cmp['f1_diff_ci'][0]:+.3f}, {hetero_cmp['f1_diff_ci'][1]:+.3f}]{sig}")

results["graph_stats"] = {"n_nodes": hg.n_nodes, "n_papers": len(paper_abstracts), "n_methods": n_methods,
    "n_datasets": n_datasets, "n_edges": len(hg.edges), "n_positive_pairs": len(pos_node_pairs)}

with open(OUT_DIR / "hetero_graph_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
torch.save(model.state_dict(), OUT_DIR / "rgcn_lite.pt")
print("\nSaved to", OUT_DIR / "hetero_graph_results.json", "and", OUT_DIR / "rgcn_lite.pt")
